<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

## **Local Setup**

Run the two cells below to verify that the correct environment loaded. Ignore this if you're on Google Colab session.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

executable_path = Path(sys.executable)
active_env_idx = executable_path.parts.index("envs")
assert executable_path.parts[active_env_idx+1] == "default", "Wrong environment. Choose 'default' environment."
print("'Default' environment is loaded successfully!")

## **Google Colab Setup**

Please uncomment the cell below and run it to install all dependencies and resolve import paths. After installation is completed, restart your session to clear Colab's previous Python cache.

**CPU vs. GPU**: To install CUDA-enabled PyTorch, start your session with GPU runtime session and replace `uv pip install -e .[cpu,dev] ...` with `uv pip install -e .[gpu,dev] ...` in the fillowing cell.  

In [ ]:
# # Clone the repo locally
# !rm -rf /content/build-a-llm-from-scratch-book
# !git clone https://github.com/paymantohidifar/build-a-llm-from-scratch-book.git --branch main
# %cd build-a-llm-from-scratch-book/llms-from-scratch

# # Bootstrap uv globally and pull GPU-enabled binaries directly into the system layer
# !curl -LsSf https://astral.sh/uv/install.sh | sh && \
# export PATH="$HOME/.local/bin:${PATH}" && \
# uv pip install -e .[gpu,dev] \
#         --system \
#         --break-system-packages \
#         --color never

# # Add `src/` to system path for local imports
# import sys
# sys.path.append('/content/build-a-llm-from-scratch-book/llms-from-scratch/src') # Point to the cloned directory

## **Verify Installation of Necessary Packages**

# **Chapter 2: Working with Text Data**

---

## **Table of Contents**

- [Introduction](#introduction)
- [2.1 Understanding word embeddings](#21-understanding-word-embeddings)
- [2.2 Tokenizing text](#22-tokenizing-text)
- [2.3 Converting tokens into token IDs](#23-converting-tokens-into-token-ids)
- [2.4 Adding special context tokens](#24-adding-special-context-tokens)
- [2.5 BytePair encoding](#25-bytepair-encoding)
  - [Exercise 2.1: Byte pair encoding of unknown words](#exercise-21-byte-pair-encoding-of-unknown-words)
  - [Solution 2.1](#solution-21)
- [2.6 Data sampling with a sliding window](#26-data-sampling-with-a-sliding-window)
- [2.7 Creating token embeddings](#27-creating-token-embeddings)
- [2.8 Encoding word positions](#28-encoding-word-positions)
- [Summary and takeaways](#summary-and-takeaways)
- [Supplementary Materials](#supplementary-materials)

---

## **Introduction**

In this boook, we focus on decoder-only, transformer-based LLMs (the architecture behind ChatGPT). During pretraining, these models learn from vast amounts of unlabeled text via a next-word prediction task, building foundation capabilities that can later be fine-tuned for instructions or classification tasks.

To prepare input text for an LLM, the raw text must be converted into a machine-readable format. This involves splitting text into individual tokens using advanced tokenization schemes like *Byte-Pair Encoding (BPE)*—the standard algorithm utilized in modern GPT models—before structuring them into input-output pairs using a data-sampling pipeline.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" style='width:100%'>
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.1</strong> The three main stages of coding an LLM. This chapter focuses on step 1 of stage 1: implementing the data sample pipeline.
  </figcaption>
</figure>

---

## **2.1 Understanding word embeddings**

This section explains that deep neural networks cannot process raw categorical text because it is mathematically incompatible with the operations used to train them. To bridge this gap, text must be converted into continuous-valued vectors, a process called **embedding**. At its core, an embedding maps discrete objects, such as words or images, into points in a continuous vector space.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="100%">
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.2</strong> Deep learning models cannot process data formats like video, audio, and text in their raw form. Thus, we use an embedding model to transform this raw data into a dense vector representation that deep learning architectures can easily understand and process. Specifically, this figure llustrates the process of converting raw data into a three-dimensional numerical vector.
  </figcaption>
</figure>

The section details several key aspects of how these representations work:

*   **Semantic Similarity**: Drawing on earlier algorithms like **Word2Vec**, the source explains that words appearing in similar contexts tend to have similar meanings. Consequently, when these words are projected into a vector space, related terms cluster together. The figure below illustrates a 2-dimensional embedding space (t-SNE plot).
*   **Dimensionality Tradeoffs**: Word embeddings can range from a few to thousands of dimensions. While higher dimensionality captures more nuanced linguistic relationships, it represents a direct tradeoff with computational efficiency. For example, the smallest GPT-2 models use 768 dimensions, while the largest GPT-3 models use 12,288.
*   **Learned Representations**: Unlike traditional approaches that might use fixed pretrained vectors, LLMs typically learn and optimize their own embeddings during training. This ensures that the vector representations are specifically optimized for the model's particular task and dataset. 

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="100%">
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.3</strong> If word embeddings are two-dimensional, we can plot them in a two-dimensional scatterplot for visualization purposes as shown here. When using word embedding techniques, such as Word2Vec, words corresponding to similar concepts often appear close to each other in the embedding space. For instance, different types of birds appear closer to each other in the embedding space than in countries and cities.
  </figcaption>
</figure>

Ultimately, embeddings provide the dense numerical representation necessary for deep learning architectures to understand and process human language.

---

## **2.2 Tokenizing text**

This section details the essential preprocessing step of splitting raw input text into smaller units called **tokens**, which is a prerequisite for creating the numerical embeddings used by LLMs.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="100%">
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.4</strong> A view of the text processing steps in the context of an LLM. Here, we split an input text into individual tokens, which are either words or special characters, such as punctuation characters.
  </figcaption>
</figure>

The core concepts and techniques discussed in this section include:

* **Definition and Types**: Tokenization involves breaking down a sequence of text into individual components, which can be words or special characters, such as punctuation.
* **Preserving Capitalization**: Unlike some traditional NLP tasks that convert all text to lowercase, this implementation retains capitalization. This is critical because it helps the LLM distinguish between common and proper nouns, understand sentence boundaries, and learn to generate text with correct grammar.
* **Regular Expression Processing**: Using Python’s `re` library, the section demonstrates how a simple split by whitespace often leaves punctuation attached to words (e.g., "Hello," instead of "Hello" and ","). A more effective tokenizer uses specialized regular expressions to ensure punctuation marks are treated as separate tokens.
* **Handling Whitespace**: The section explores the tradeoff of removing versus keeping whitespace. While removing whitespace reduces memory and computational requirements, keeping it is vital for models trained on structured text, such as Python code, which relies on indentation.
* **Resulting Output**: By the end of the section, a basic tokenizer is established that can process a text (such as a short story) into a neatly separated list of strings encompassing words, dashes, and various punctuation marks.

Ultimately, this process transforms unstructured "raw" text into a structured list of tokens that serves as the foundation for the next stage: converting these strings into integer-based **token IDs**.

The following section demonstrates downloading a text file, reading it into Python string object and using Python's regular expression `re` library to split the text into tokens.
* **Source text:** [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

# """
# import os
# import urllib.request

# if not os.path.exists("the-verdict.txt"):
#     url = ("https://raw.githubusercontent.com/rasbt/"
#            "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
#            "the-verdict.txt")
#     file_path = "the-verdict.txt"
#     urllib.request.urlretrieve(url, file_path)
# """

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above. The goal is to tokenize and embed this text for an LLM:

In [ ]:
import re

# The following regular expression will split on whitespaces
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [ ]:
result = re.split(r'([,.]|\s)', text)

print(result)

As we can see, this creates empty strings, let's remove them

In [ ]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

This is pretty good, and we are now ready to apply this tokenization to the raw text (**Fig. 2.5**):

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="100%">
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.5</strong> The tokenization scheme we implemented so far splits text into individual words and punctuation characters. In this specific example, the sample text gets split into 10 individual tokens.
  </figcaption>
</figure>

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

Let's calculate the total number of tokens

In [ ]:
print(len(preprocessed))

---

## **2.3 Converting tokens into token IDs**

This section describes the necessary intermediate step of mapping processed text tokens into unique integer representations before they can be transformed into embedding vectors.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.6</strong> We build a vocabulary by tokenizing the entire text in a training dataset into individual tokens. These individual tokens are then sorted alphabetically, and duplicate tokens are removed. The unique tokens are then aggregated into a vocabulary that defines a mapping from each unique token to a unique integer value. The depicted vocabulary is purposely small and contains no punctuation or special characters for simplicity.
    </figcaption>
</figure>

The primary components and challenges of this process include:

* **Building a Vocabulary**: To convert strings to numbers, a vocabulary is constructed by identifying all unique tokens in the training dataset, sorting them alphabetically, and assigning each a unique integer, known as a **token ID**. This vocabulary serves as a mapping dictionary for the model.
* **Bidirectional Mapping**: `SimpleTokenizerV1` class is implemented that features two core methods: `encode`, which splits text and converts it into token IDs, and `decode`, which uses an inverse vocabulary to turn those IDs back into human-readable text.
* **The "Unknown Word" Problem**: A significant limitation of this basic implementation is that it relies strictly on the words present in its initial training data. If the tokenizer encounters a word not included in its vocabulary, it triggers a `KeyError`, highlighting the need for larger training sets or specialized tokens to handle out-of-vocabulary terms.

Ultimately, this stage transforms the list of processed strings into a structured numerical format that mathematically-driven neural networks can begin to interpret.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.7</strong> Starting with a new text sample, we tokenize the text and use the vocabulary to convert the text tokens into token IDs. The vocabulary is built from the entire training set and can be applied to the training set itself and any new text samples. The depicted vocabulary contains no punctuation or special characters for simplicity.
    </figcaption>
</figure>

From these tokens, we can now build a vocabulary that consists of all the unique tokens

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

In [ ]:
vocab = {token:integer for integer,token in enumerate(all_words)}

Below are the first 50 entries in this vocabulary:

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

Putting it now all together into a tokenizer class:

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.8</strong> Tokenizer implementations share two common methods: an encode method and a decode method. The encode method takes in the sample text, splits it into individual tokens, and converts the tokens into token IDs via the vocabulary. The decode method takes in token IDs, converts them back into text tokens, and concatenates the text tokens into natural text.
    </figcaption>
</figure>

We can use the tokenizer to encode (that is, tokenize) texts into integers. These integers can then be embedded (later) as input of/for the LLM.

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

We can decode the integers back into text:

In [ ]:
tokenizer.decode(ids)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## **2.4 Adding special context tokens**

This section focuses on enhancing a tokenizer's ability to handle unknown terms and delineate boundaries between unrelated text sources.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.9</strong> We add special tokens to a vocabulary to deal with certain contexts. For instance, we add an <|unk|> token to represent new and unknown words that were not part of the training data and thus not part of the existing vocabulary. Furthermore, we add an <|endoftext|> token that we can use to separate two unrelated text sources.
    </figcaption>
</figure>

The primary concepts discussed include:

* **Handling Unknown Words**: The section introduces the `<|unk|>` token to represent out-of-vocabulary words encountered during inference that were not present in the initial training data.
* **Marking Document Boundaries**: To help an LLM understand when concatenated training texts are unrelated, an `<|endoftext|>` token is inserted between independent documents, such as different books or articles.
* **Other Special Tokens**: The source notes that while some models use specific markers like `[BOS]` (beginning of sequence), `[EOS]` (end of sequence), and `[PAD]` (padding), GPT models typically simplify this by using `<|endoftext|>` for all these purposes.
* **Tokenizer Update**: These improvements are implemented in a new `SimpleTokenizerV2` class, which updates the vocabulary and includes logic to map unknown strings to the `<|unk|>` ID instead of throwing errors.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.10</strong> When working with multiple independent text source, we add <|endoftext|> tokens between these texts. These <|endoftext|> tokens act as markers, signaling the start or end of a particular segment, allowing for more effective processing and understanding by the LLM.
    </figcaption>
</figure>

Notably, the section concludes by mentioning that GPT models do not actually use an `<|unk|>` token; instead, they utilize a more advanced approach called **Byte Pair Encoding (BPE)** to handle any word by breaking it down into subword units and will be covered in next section.

Let's see what happens if we tokenize the following text:

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

The above produces an error because the word "Hello" is not contained in the vocabulary. To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words. Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text.

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token:

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Let's try to tokenize text with the modified tokenizer:

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## **2.5 BytePair encoding**

This section introduces a more sophisticated tokenization scheme used by models such as GPT-2, GPT-3, and ChatGPT. 

The key features and advantages of this method include:

* **Handling Unknown Words**: Unlike the basic tokenizers discussed in previous sections, BPE can process any word by breaking it down into subword units or individual characters. This eliminates the need for an `<|unk|>` token for out-of-vocabulary terms.
* **Iterative Merging Algorithm**: The BPE vocabulary is built by starting with individual characters and iteratively merging the most frequent character combinations into subwords and eventually full words. For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges.
* **Standard Library**: For practical implementation, OpenAI’s open-source `tiktoken` library is used. The library efficiently implements BPE in Rust to improve computational performance. The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py).
* **Vocabulary Context**: The BPE tokenizer for GPT-2 and GPT-3 has a total vocabulary size of 50,257, with the `<|endoftext|>` marker assigned the largest ID.

Ultimately, BPE provides the robust foundation necessary for modern LLMs to understand and generate text across a nearly infinite variety of linguistic inputs.

>**Bonus:** The author has provided a [bonus notebook](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb), where he compared different implementations of BPE side-by-side (tiktoken was about 5x faster on the sample text).

In [ ]:
# Define a sample text
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

# Instantiate a GPT2 (BPE) tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Encode text
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# Decode text
strings = tokenizer.decode(integers)
print(strings)

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.11</strong> BPE tokenizers break down unknown words into subwords and individual characters. This way, a BPE tokenizer can parse any word and doesn’t need to replace unknown words with special tokens, such as <|unk|>.
    </figcaption>
</figure>

### **Exercise 2.1: Byte pair encoding of unknown words**

Try the BPE tokenizer from the tiktoken library on the unknown words “Akwirw ier” and print the individual token IDs. Then, call the decode function on each of the resulting integers in this list to reproduce the mapping shown in above figure. Lastly, call the decode method on the token IDs to check whether it can reconstruct the original input, “Akwirw ier.”

### **Solution 2.1:**

In [ ]:
word = "Akwirw ier"
ex2_1_integers = tokenizer.encode(word)
print(ex2_1_integers)
assert ex2_1_integers == [33901, 86, 343, 86, 220, 959]

ex2_1_string = tokenizer.decode(ex2_1_integers)
print(ex2_1_string)
assert ex2_1_string == word

---

## **2.6 Data sampling with a sliding window**

This section details the process of generating the input-target pairs essential for pretraining an LLM to perform next-word prediction.

The core concepts of this sampling process include:

* **Next-Word Prediction Logic**: To train the model, the dataset is organized into input sequences ($X$) and target sequences ($y$), where the target is simply the input sequence shifted forward by one token (**Fig. 2.12**). For every token in the input, the model's task is to predict the token that immediately follows it in the original text.
* **Sliding Window Mechanism**: This approach involves moving a fixed-size window (defined by `max_length` parameter) across the tokenized training data to extract these sequences.
* **The Role of Stride**: The `stride` parameter determines the distance the window moves between consecutive samples. While a small stride (like 1) creates heavily overlapping batches for demonstration, setting the stride equal to the `max_length` prevents overlap, which can be useful to avoid overfitting during training.
* **Efficient Batching with PyTorch**: The implementation utilizes PyTorch’s `Dataset` and `DataLoader` classes to automate this process. A custom `GPTDatasetV1` class is created to chunk the text into tensors, and the `DataLoader` handles higher-level tasks like shuffling and organizing the data into batches for the training loop.

<figure style="text-align: center; width: 600px; margin: 0 auto;">
  <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="100%">
  <figcaption style="text-align: left; margin-top: 8px;">
    <strong>Figure 2.12</strong> Given a text sample, extract input blocks as subsamples that serve as input to the LLM, and the LLM’s prediction task during training is to predict the next word that follows the input block. During training, we mask out all words that are past the target. Note that the text shown in this figure must undergo tokenization before the LLM can process it; however, this figure omits the tokenization step for clarity.
  </figcaption>
</figure>

This stage is the final step in the data preparation pipeline, converting token IDs into a structured format ready to be transformed into numerical embedding vectors. 

Let's deep-dive into the code. First, define tokenizer and read raw text into a long string object:

In [ ]:
import tiktoken


tokenizer = tiktoken.get_encoding('gpt2')

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

We remove the first 50 tokens from the dataset for demonstration purposes, as it results in a slightly more interesting text passage in the next steps:

In [ ]:
enc_sample = enc_text[50:]

For each text chunk, we want the inputs and targets. Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right:

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

One by one, the prediction would look like as follows:

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

We will take care of the next-word prediction in a later chapter after we covered the attention mechanism. For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one:

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

We use a sliding window approach, changing the position by +1:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.13</strong> To implement efficient data loaders, we collect the inputs in a tensor, x, where each row represents one input context. A second tensor, y, contains the corresponding prediction targets (next words), which are created by shifting the input by one position.
    </figcaption>
</figure>

Next, we create dataset and dataloader that extract chunks from the input text dataset and output a Python iterable factory of mini-batches. We can create mini-batch stream using Python's `iter` and interate throguh it:

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

An example using stride equal to the context length (here: 4) as shown below:

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.14</strong>  When creating multiple batches from the input dataset, we slide an input window across the text. If the stride is set to 1, we shift the input window by one position when creating the next batch. If we set the stride equal to the input window size, we can prevent overlaps between the batches.
    </figcaption>
</figure>

We can also create batched outputs. Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting:

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

---

## **2.7 Creating token embeddings**

Section 2.7 describes the final stage of the data preparation pipeline: transforming discrete token IDs into continuous numerical vectors that a neural network can process (compatible for back-propagation).

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.15</strong> Preparation involves tokenizing text, converting text tokens to token IDs, and converting token IDs into embedding vectors. Here, we consider the previously created token IDs to create the token embedding vectors.
    </figcaption>
</figure>

The key technical and conceptual details from this section include:

* **Necessity for Deep Learning**: Because GPT-like LLMs are trained using the backpropagation algorithm, they require continuous vector representations rather than categorical integers. These embeddings allow the model to represent words in a mathematical space where it can calculate gradients and update weights.
* **The Lookup Mechanism**: An embedding layer in PyTorch functions essentially as a lookup table. The token ID serves as an index to retrieve a specific row from a high-dimensional weight matrix. For example, if a token ID is 5, the model retrieves the sixth row of the embedding matrix (using zero-based indexing) to use as that token's vector representation (**Fig. 2.16**).
* **Weight Initialization and Optimization**: The embedding weight matrix is initially filled with small, random values. These values are not fixed; they are trainable parameters that the LLM refines during the training process to better capture semantic meanings and relationships between tokens.
* **Computational Efficiency**: The source notes that while using an embedding layer is conceptually identical to performing matrix multiplication on one-hot encoded vectors (see [embeddins-and-linear-layers.ipynb](../03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb) for details), the lookup approach is far more computationally efficient and optimized for deep learning frameworks.
* **Dimensionality**: LLMs use large embedding spaces. For instance, the GPT-3 model utilizes an embedding size of 12,288 dimensions to capture complex linguistic nuances.

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.16</strong> Embedding layers perform a lookup operation, retrieving the embedding vector corresponding to the token ID from the embedding layer’s weight matrix. For instance, the embedding vector of the token ID 5 is the sixth row of the embedding layer weight matrix. We assume that the token IDs were produced by the small vocabulary from section 3.
    </figcaption>
</figure>

Ultimately, this process converts the structured list of integers into a dense numerical format, providing the foundation for the model's layers to begin interpreting human language. Let's see how we create embedding matrix using PyTorch. Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

This would result in a 6x3 weight matrix:

In [ ]:
print(embedding_layer.weight)

To convert a token with id 3 into a 3-dimensional vector, we do the following:

In [ ]:
print(embedding_layer(torch.tensor([3])))

Note that the above is the 4th row in the `embedding_layer` weight matrix. To embed all four `input_ids` values above, we do:

In [ ]:
print(embedding_layer(input_ids))

**You may be interested in the bonus content comparing embedding layers with regular linear layers: [embeddins-and-linear-layers.ipynb](../03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb)**

---

## **2.8 Encoding word positions**

This section addresses a fundamental limitation of the self-attention mechanism: it is position-agnostic and cannot inherently distinguish the order or position of tokens in a sequence.

The key concepts and implementation details for solving this include:

* **The Need for Positional Information**: Because standard embedding layers map the same token ID to the same vector regardless of where it appears (**Fig. 2.17**), an LLM requires additional information to understand sentence structure and word order.

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.17</strong>  The embedding layer converts a token ID into the same vector representation regardless of where it is located in the input sequence. For example, the token ID 5, whether it’s in the first or fourth position in the token ID input vector, will result in the same embedding vector.
    </figcaption>
</figure>

* **Types of Positional Embeddings**: 
    * **Absolute Positional Embeddings**: These are tied to specific locations in a sequence (e.g., the 1st, 2nd, or 3rd token).
    * **Relative Positional Embeddings**: These focus on the distance between tokens, allowing a model to generalize better to sequences of varying lengths.
* **The GPT Approach**: OpenAI’s GPT models utilize absolute positional embeddings. Unlike the fixed sinusoids used in the original Transformer architecture, these embeddings are trainable parameters optimized during the model's learning process.
* **Implementation**: In PyTorch, positional embeddings are created as a second embedding layer of the same dimensionality as the token embeddings. The input to this layer is a placeholder vector representing the sequence indices ($0, 1, \dots, \text{max\_length} - 1$).
* **Final Input Vector**: The resulting positional embedding vectors are added directly to the token embedding vectors. This combined representation is then passed into the main LLM modules, ensuring the model can process both the semantic meaning and the sequence order of the text.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.18</strong>  Positional embeddings are added to the token embedding vector to create the input embeddings for an LLM. The positional vectors have the same dimension as the original token embeddings. The token embeddings are shown with value 1 for simplicity.
    </figcaption>
</figure>

The BytePair encoder has a vocabulary size of 50,257. Suppose we want to encode the input tokens into a 256-dimensional vector representation:

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

We sample data from the dataloader and embed the tokens in each batch into a 256-dimensional vector. For example, if we have a batch size of 8 with 4 tokens each, this results in a $8 \times 4 \times 256$ tensor:

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(token_embeddings)

GPT-2 uses absolute position embeddings, so we just create another embedding layer:

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
print(pos_embedding_layer.weight)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

In the initial phase of the input processing workflow, the input text is segmented into separate tokens. Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="100%">
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 2.19</strong>  As part of the input processing pipeline, input text is first broken up into individual tokens. These tokens are then converted into token IDs using a vocabulary. The token IDs are converted into embedding vectors to which positional embeddings of a similar size are added, resulting in input embeddings that are used as input for the main LLM layers.
    </figcaption>
</figure>

---

## **Summary and takeaways**

* LLMs require textual data to be converted into numerical vectors, known as embeddings, since they can’t process raw text. Embeddings transform discrete
data (like words or images) into continuous vector spaces, making them compatible with neural network operations.
* As the first step, raw text is broken into tokens, which can be words or characters. Then, the tokens are converted into integer representations, termed token IDs.
* Special tokens, such as `<|unk|>` and `<|endoftext|>`, can be added to enhance the model’s understanding and handle various contexts, such as unknown words or marking the boundary between unrelated texts.
* The byte pair encoding (BPE) tokenizer used for LLMs like GPT-2 and GPT-3 can efficiently handle unknown words by breaking them down into subword units or individual characters.sss
s* Embedding layers in PyTorch function as a lookup operation, retrieving vectors corresponding to token IDs. The resulting embedding vectors provide continuous       s of tokens, whi      ch is crucial for training deep learning models like LLMs.
* While token embeddings provide consistent vector representations for each token, they lack a sense of the token’s position in a sequence. To rectify this, two main types of positional embeddings exist: absolute and relative. OpenAI’s GPT models utilize absolute positional embeddings, which are added to the token embedding vectors and are optimized during the model training.

---

## **Supplementary Materials**

* See the [dataloader.ipynb](./dataloader.ipynb) code notebook, which is a concise version of the data loader that we implemented in this chapter and will need for training the GPT model in upcoming chapters.
* See the [Byte Pair Encoding (BPE) Tokenizer From Scratch](../05_bpe-from-scratch/) if you are interested in learning how the GPT-2 tokenizer can be implemented and trained from scratch.
    * [bpe-from-scratch-simple.ipynb](bpe-from-scratch-simple.ipynb) contains optional (bonus) code that explains and shows how the BPE tokenizer works under the hood; this is geared for simplicity and readability.
    * [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) implements a more sophisticated (and much more complicated) BPE tokenizer that behaves similarly as tiktoken with respect to all the edge cases; it also has additional funcitionality for loading the official GPT-2 vocab.
* OpenAI's original BPE tokenizer source code can be found [here](../02_bonus_bytepair-encoder/).
* See bonus content comparing embedding layers with regular linear layers: [embeddins-and-linear-layers.ipynb](../03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb)